In [6]:
%%writefile train_gemini_3_1_sota.py
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" # <-- ADD THIS LINE
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from transformers import (
    AutoTokenizer, AutoProcessor, AutoFeatureExtractor,
    AutoModel, AutoModelForCausalLM 
)
from datasets import load_dataset
from accelerate import Accelerator
from accelerate.state import AcceleratorState
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# =========================================================
# 2026 KAGGLE CUDA MULTIPROCESSING FIX
# =========================================================
import torch.multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

# 🌟 2026 SOTA BACKBONE: NATIVE GQA + DEEPSEEK REASONING
LLM_BACKBONE = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B" 

# ==========================================
# 1. SHARDED MULTIMODAL INGESTOR
# ==========================================
class Gemini3DataStreamer(IterableDataset):
    def __init__(self, mode="sft", max_length=128, max_audio_sec=5):
        self.mode = mode
        self.max_length = max_length
        self.max_audio_samples = 16000 * max_audio_sec 
        
        self.tokenizer = AutoTokenizer.from_pretrained(LLM_BACKBONE)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        # SOTA 2026 Encoders (SigLIP > CLIP)
        self.v_proc = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
        self.a_proc = AutoFeatureExtractor.from_pretrained("openai/whisper-small")
        
        self.v_stream = load_dataset("HuggingFaceH4/llava-instruct-mix-vsft", split="train", streaming=True)
        self.a_stream = load_dataset("PolyAI/minds14", "en-US", split="train", streaming=True)
        if mode == "dpo":
            self.dpo_stream = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train", streaming=True)

    def _extract_text(self, item):
        if isinstance(item, str): return item
        if isinstance(item, list) and len(item) > 0:
            if isinstance(item[0], dict) and 'text' in item[0]: return " ".join([i['text'] for i in item if 'text' in i])
        return str(item)

    def __iter__(self):
        try:
            state = AcceleratorState()
            process_index, num_processes = state.process_index, state.num_processes
        except Exception:
            process_index, num_processes = 0, 1

        v_iter, a_iter = iter(self.v_stream), iter(self.a_stream)
        d_iter = iter(self.dpo_stream) if self.mode == "dpo" else None
        step = 0
        
        while True:
            try:
                v_item, a_item = next(v_iter), next(a_iter)
                
                # SOTA Multi-GPU Dataloader Sharding
                if step % num_processes != process_index:
                    step += 1
                    if self.mode == "dpo": next(d_iter)
                    continue
                step += 1
                
                img = v_item.get('image') or (v_item.get('images')[0] if v_item.get('images') else None)
                if img is None: continue
                pixels = self.v_proc(images=img.convert("RGB"), return_tensors="pt")['pixel_values'].squeeze(0)
                
                audio_array = a_item['audio']['array']
                if len(audio_array) > self.max_audio_samples: audio_array = audio_array[:self.max_audio_samples]
                else: audio_array = np.pad(audio_array, (0, self.max_audio_samples - len(audio_array)))
                audio = self.a_proc(audio_array, sampling_rate=16000, return_tensors="pt")['input_features'].squeeze(0)

                if self.mode == "sft":
                    raw_text = next((m['content'] for m in reversed(v_item.get('messages', [])) if m['role'] == 'assistant'), "")
                    if not raw_text and 'conversations' in v_item: raw_text = v_item['conversations'][-1]['value']
                    text_str = self._extract_text(raw_text)
                    if not text_str: continue
                    tokens = self.tokenizer(text_str, max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    yield pixels, tokens, audio
                else: 
                    d_item = next(d_iter)
                    c_tokens = self.tokenizer(d_item['chosen'], max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    r_tokens = self.tokenizer(d_item['rejected'], max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    yield pixels, c_tokens, r_tokens, audio
            except StopIteration: break
            except Exception: continue

# ==========================================
# 2. GEMINI 3.1 / DEEPSEEK ARCHITECTURE
# ==========================================
class MultimodalGLU(nn.Module):
    """SOTA 2026 Projector using Gated Linear Units (SwiGLU paradigm)"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj_v = nn.Linear(in_dim, out_dim, bias=False)
        self.proj_gate = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, x): 
        return self.proj_v(x) * F.silu(self.proj_gate(x))

class Top2SparseMoE(nn.Module):
    """Mixture of Experts applied to multimodal embeddings before LLM ingestion"""
    def __init__(self, d_model, num_experts=4, top_k=2):
        super().__init__()
        self.num_experts, self.top_k = num_experts, top_k
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        self.experts = nn.ModuleList([MultimodalGLU(d_model, d_model) for _ in range(num_experts)])
        
    def forward(self, x):
        b, s, d = x.shape
        flat_x = x.view(-1, d)
        logits = self.gate(flat_x)
        scores = F.softmax(logits, dim=-1)
        weights, indices = torch.topk(scores, self.top_k, dim=-1)
        weights = weights / weights.sum(dim=-1, keepdim=True) 
        
        out = torch.zeros_like(flat_x)
        for i, expert in enumerate(self.experts):
            mask = indices == i
            for k in range(self.top_k):
                expert_mask = mask[:, k]
                if expert_mask.any(): out[expert_mask] += expert(flat_x[expert_mask]) * weights[expert_mask, k].unsqueeze(1)
        
        density = scores.mean(dim=0)
        aux_loss = (density * density).sum() * self.num_experts
        return out.view(b, s, d), aux_loss

class Gemini3_1_Tiny(nn.Module):
    def __init__(self, compute_dtype=torch.bfloat16):
        super().__init__()
        self.compute_dtype = compute_dtype
        
        # 🌟 NATIVE SDPA: Bypasses flash_attn pip install but retains 100% of the speed
        self.llm = AutoModelForCausalLM.from_pretrained(
            LLM_BACKBONE, 
            torch_dtype=compute_dtype, 
            attn_implementation="sdpa" 
        )
        d_model = self.llm.config.hidden_size
        
        self.vit = AutoModel.from_pretrained("google/siglip-base-patch16-224", torch_dtype=compute_dtype).vision_model
        self.audio_enc = AutoModel.from_pretrained("openai/whisper-small", torch_dtype=compute_dtype).get_encoder()
        
        self.llm.requires_grad_(False)
        self.vit.requires_grad_(False)
        self.audio_enc.requires_grad_(False)
        self.llm.gradient_checkpointing_enable()
        self.v_proj = MultimodalGLU(self.vit.config.hidden_size, d_model).to(compute_dtype)
        self.a_proj = MultimodalGLU(self.audio_enc.config.d_model, d_model).to(compute_dtype)
        self.moe = Top2SparseMoE(d_model, num_experts=4, top_k=2).to(compute_dtype)
        self.norm = nn.LayerNorm(d_model, dtype=compute_dtype)

    def forward(self, tokens, pixels, audio, pad_token_id):
        pixels, audio = pixels.to(self.compute_dtype), audio.to(self.compute_dtype)
        
        with torch.no_grad():
            v_feat = self.vit(pixels).last_hidden_state
            a_feat = self.audio_enc(audio).last_hidden_state
        
        # SOTA: Spatial Vision Compression (Dynamic Grid Pooling)
        B, L, D = v_feat.shape
        grid = int(L ** 0.5)
        if grid * grid == L:
            v_feat = F.avg_pool2d(v_feat.view(B, grid, grid, D).permute(0, 3, 1, 2), 2, 2).permute(0, 2, 3, 1).reshape(B, -1, D)
            
        a_feat = F.avg_pool1d(a_feat.transpose(1, 2), 2, 2).transpose(1, 2)
        v_embeds, a_embeds = self.v_proj(v_feat), self.a_proj(a_feat)
        
        mm_embeds = torch.cat([a_embeds, v_embeds], dim=1)
        mm_moe_out, aux_loss = self.moe(self.norm(mm_embeds))
        mm_embeds = mm_embeds + mm_moe_out # Residual connection
        
        t_embeds = self.llm.get_input_embeddings()(tokens).to(self.compute_dtype)
        inputs_embeds = torch.cat([mm_embeds, t_embeds], dim=1)
        
        mm_mask = torch.ones((tokens.size(0), mm_embeds.size(1)), dtype=torch.long, device=tokens.device)
        text_mask = (tokens != pad_token_id).long()
        attention_mask = torch.cat([mm_mask, text_mask], dim=1)
        
        outputs = self.llm(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
        return outputs.logits, aux_loss, mm_embeds.size(1)

# ==========================================
# 3. LOSS & DISTRIBUTED TRAINING
# ==========================================
def compute_causal_loss(logits, tokens, mm_len, pad_token_id):
    labels = tokens.clone()
    labels[labels == pad_token_id] = -100
    mm_labels = torch.full((tokens.size(0), mm_len), -100, dtype=torch.long, device=tokens.device)
    full_labels = torch.cat([mm_labels, labels], dim=1)
    return F.cross_entropy(logits[..., :-1, :].contiguous().view(-1, logits.size(-1)), full_labels[..., 1:].contiguous().view(-1))

def get_batch_logps(logits, tokens, mm_len, pad_token_id):
    labels = tokens.clone()
    labels[labels == pad_token_id] = -100
    mm_labels = torch.full((tokens.size(0), mm_len), -100, dtype=torch.long, device=tokens.device)
    full_labels = torch.cat([mm_labels, labels], dim=1)
    shift_logits, shift_labels = logits[..., :-1, :].contiguous(), full_labels[..., 1:].contiguous()
    log_probs = F.log_softmax(shift_logits, dim=-1)
    loss_mask = shift_labels != -100
    shift_labels[shift_labels == -100] = 0 
    return (torch.gather(log_probs, dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1) * loss_mask).sum(dim=-1)

def main():
    acc = Accelerator(mixed_precision="bf16")
    acc.print(">>> Initializing Vision-R1 SOTA Architecture (Native SDPA + GQA)...")
    
    tokenizer = AutoTokenizer.from_pretrained(LLM_BACKBONE)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    
    model = Gemini3_1_Tiny(compute_dtype=torch.bfloat16)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    
    sft_loader = DataLoader(Gemini3DataStreamer(mode="sft"), batch_size=1)
    model, opt, sft_loader = acc.prepare(model, opt, sft_loader)
    
    acc.print("\n>>> Phase 1: SFT (Multimodal Alignment)...")
    model.train()
    for i, (p, t, a) in enumerate(sft_loader):
        opt.zero_grad()
        logits, aux_loss, mm_len = model(t, p, a, tokenizer.pad_token_id)
        loss = compute_causal_loss(logits, t, mm_len, tokenizer.pad_token_id)
        
        acc.backward(loss + 0.01 * aux_loss)
        opt.step()
        if i % 10 == 0: acc.print(f"SFT Step {i} | Loss: {loss.item():.4f}")
        if i >= 100: break 

    import gc
    gc.collect()
    torch.cuda.empty_cache()

    dpo_loader = DataLoader(Gemini3DataStreamer(mode="dpo"), batch_size=1)
    dpo_loader = acc.prepare(dpo_loader)
    
    acc.print("\n>>> Phase 2: DPO (Deep Reasoning Alignment)...")
    for i, (p, c_t, r_t, a) in enumerate(dpo_loader):
        opt.zero_grad()
        logits_c, _, mm_len = model(c_t, p, a, tokenizer.pad_token_id)
        logits_r, _, _ = model(r_t, p, a, tokenizer.pad_token_id)
        
        l_c = get_batch_logps(logits_c, c_t, mm_len, tokenizer.pad_token_id)
        l_r = get_batch_logps(logits_r, r_t, mm_len, tokenizer.pad_token_id)
        dpo_loss = -F.logsigmoid(0.1 * (l_c - l_r)).mean()
        
        acc.backward(dpo_loss)
        opt.step()
        if i % 10 == 0: acc.print(f"DPO Step {i} | Loss: {dpo_loss.item():.4f}")
        if i >= 50: break

    acc.wait_for_everyone()
    if acc.is_main_process:
        unwrap = acc.unwrap_model(model)
        torch.save({
            "v_proj": unwrap.v_proj.state_dict(),
            "a_proj": unwrap.a_proj.state_dict(),
            "moe": unwrap.moe.state_dict(),
        }, "gemini_3_1_adapters.pth")
        acc.print(">>> Training Complete! SOTA Adapters Saved.")

if __name__ == "__main__":
    main()

Overwriting train_gemini_3_1_sota.py


In [7]:
!accelerate launch --multi_gpu --num_processes=2 train_gemini_3_1_sota.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/local/lib/python3.12/dist-packages/accelerate/utils/launch.py:238: UserWarning: Port `29500` is already in use. Accelerate will attempt to launch in a standalone-like mode by finding an open port automatically for this session. If this current attempt fails, or for more control in future runs, please specify a different port (e.g., `--main_process_port <your_chosen_port>`) or use `--main_process_port 0` for automatic selection in your launch command or Accelerate config file.
  warnings.warn(
[W325 07:23:49.627477107 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W325 07:23:57.494311314 socket.cpp:207] [c10d] T

In [11]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoProcessor, AutoFeatureExtractor
from accelerate import dispatch_model, infer_auto_device_map
from PIL import Image
import requests
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from train_gemini_3_1_sota import Gemini3_1_Tiny, LLM_BACKBONE

print(">>> Booting up Vision-R1 (DeepSeek) Inference Engine...")

tok = AutoTokenizer.from_pretrained(LLM_BACKBONE)
model = Gemini3_1_Tiny(compute_dtype=torch.bfloat16)

checkpoint = torch.load("/kaggle/working/gemini_3_1_adapters.pth", map_location="cpu")
model.v_proj.load_state_dict(checkpoint['v_proj'])
model.a_proj.load_state_dict(checkpoint['a_proj'])
model.moe.load_state_dict(checkpoint['moe'])

# Pipeline Parallelism: Shards model across T4 x2 automatically
device_map = infer_auto_device_map(model, max_memory={0: "14GiB", 1: "14GiB"})
model = dispatch_model(model, device_map=device_map)
model.eval()

@torch.no_grad()
def generate_gemini_response(question, image_url):
    device = next(model.parameters()).device
    dtype = model.compute_dtype
    
    v_proc = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
    a_proc = AutoFeatureExtractor.from_pretrained("openai/whisper-small")

    # Image
    img = Image.open(requests.get(image_url, stream=True).raw).convert("RGB") if image_url.startswith("http") else Image.open(image_url).convert("RGB")
    pixels = v_proc(images=img, return_tensors="pt")['pixel_values'].to(device, dtype=dtype)
    
    # Audio (Mock Silent)
    audio = a_proc(np.zeros(16000 * 5), sampling_rate=16000, return_tensors="pt")['input_features'].to(device, dtype=dtype)
    
    # 🌟 FIX 1: ADD A STRICT ENGLISH SYSTEM PROMPT 🌟 
    prompt = (
        "<|im_start|>system\nYou are a highly advanced multimodal AI. You MUST think and respond strictly and only in the English language.<|im_end|>\n"
        f"<|im_start|>user\n<image>\n{question}<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n"
    )
    input_ids = tok.encode(prompt, return_tensors="pt").to(device)
    
    v_feat = model.vit(pixels).last_hidden_state
    a_feat = model.audio_enc(audio).last_hidden_state
    
    B, L, D = v_feat.shape
    grid = int(L ** 0.5)
    if grid * grid == L:
        v_feat = F.avg_pool2d(v_feat.view(B, grid, grid, D).permute(0, 3, 1, 2), 2, 2).permute(0, 2, 3, 1).reshape(B, -1, D)
    a_feat = F.avg_pool1d(a_feat.transpose(1, 2), 2, 2).transpose(1, 2)
    
    v_embeds, a_embeds = model.v_proj(v_feat), model.a_proj(a_feat)
    
    mm_embeds = torch.cat([a_embeds, v_embeds], dim=1)
    mm_moe_out, _ = model.moe(model.norm(mm_embeds))
    mm_embeds = mm_embeds + mm_moe_out
    
    t_embeds = model.llm.get_input_embeddings()(input_ids).to(dtype)
    inputs_embeds = torch.cat([mm_embeds, t_embeds], dim=1)
    attention_mask = torch.ones((1, inputs_embeds.size(1)), dtype=torch.long, device=device)
    
    # 🌟 FIX 2: TUNE GENERATION KWARGS TO PREVENT CHINESE FALLBACK 🌟
    out_ids = model.llm.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        max_new_tokens=300, 
        pad_token_id=tok.eos_token_id,
        do_sample=True,
        temperature=0.3,          # Lowered to stop wild hallucinations
        repetition_penalty=1.15,  # Prevents it from looping words
        top_p=0.85
    )
    return tok.decode(out_ids[0], skip_special_tokens=True)

test_image = "/kaggle/input/datasets/anwarshamim/images-for-test/images.jpg"
response = generate_gemini_response("Analyze this image step-by-step and tell me what you deduce in english language ", test_image)

print("-" * 50)
print(f" Braga'aye Dance:\n\n{response}")

>>> Booting up Vision-R1 (DeepSeek) Inference Engine...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

--------------------------------------------------
DEEPSEEK-R1 x GEMINI 3.1 PRO:

好的，我现在要分析这个图片。首先，我看到这是一个关于时间线的图表，可能显示了一些历史事件的时间安排。让我仔细看看。

图片中有一个日期标记在2018年左右，具体是2019年6月15日。然后，旁边有一段文字：“The future of the world is bright, and we are ready for change.” 这句话看起来是在强调未来的美好和我们对未来充满希望的态度。

接下来，图片上有几个关键点：一个红色方块标示了“2019年”，接着是一个箭头指向未来，显示出时间向右延伸到更远的地方。这表明时间线从现在开始继续向前发展，暗示着未来的无限可能。

再往下看，图片中有几行数字，比如“2019”、“2023”等，这些可能是具体的年份或重要时间节点。但没有更多的细节信息，所以主要关注的是文字内容和时间线的趋势。

综合来看，这张图片传达的信息应该是积极向上，强调了对未来的信心和准备状态。
</think>

根据提供的图片描述，我们可以得出以下结论：

1. **背景**：图片描绘了一个充满活力和希望的画面，展示了时间线从当前（即2019年）向未来延伸的方向发展。

2. **文字内容**：
   - 文章标题为“The Future of the World is Bright, and We Are Ready for Change。”
   - 
